In [ ]:
对每一个 tile：
    对每一个 label：
        找到这个 label 对应的所有 polygons
        如果 XML annotation name 中包含多个 comma-separated labels，则先拆分并把同一个 polygon 分配给对应的多个 labels
        对同一个 label 下的所有 polygons 构建 union annotation region
        计算 tile 与该 union annotation region 的 overlap area
        用 overlap area / tile area 作为该 label 的 soft label
soft label = Area(tile ∩ union of all polygons for this label) / Area(tile)

1. 读取 hard-label CSV
2. 读取 XML annotation polygons
3. 拆分 comma-separated multi-label annotation names
4. 将同一个 polygon 分配给所有对应的 selected labels
5. 对每个 label，将该 label 对应的所有 polygons 构建为 union annotation region
6. 对每个 tile 和每个 label，计算 tile 与该 label union region 的 overlap ratio
7. 得到每个 tile 的 12 维 continuous soft-label vector
8. 使用 soft>=0.5 与 Kee hard label 做 sanity check，但不要求 100% 完全一致
9. 生成 soft-label CSV

Kee hard label:
Kee preprocessing 基于 XML polygon annotation 和 tile overlap 生成 0/1 hard labels。
对于满足 overlap threshold 的 annotation polygon，其 annotation name 会被记录到 annotation_label 中。
若 annotation name 包含多个 comma-separated OED features，后续会被转换为多个 hard-label columns。因此 Kee 的最终 CSV 中，每个 tile 得到的是 12 个 binary labels。

Our soft label:
本研究不再只保留 threshold 后的 0/1 hard label，而是重新计算 tile-level continuous overlap ratio。
首先，XML 中 comma-separated multi-label annotation names 会被拆分；同一个 polygon 会被分配给所有对应的 selected OED features。
然后，对于每个 tile 和每个 selected feature，将该 feature 对应的所有 polygons 视为一个 annotation region，并计算 tile 与该 region 的 overlap area / tile area。最终，每个 tile 得到 12 个 0–1 continuous soft-label values。

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import xml.etree.ElementTree as ET
from PIL import Image
from tqdm import tqdm

from shapely.geometry import Polygon, box
from shapely.ops import unary_union

PROJECT_ROOT = Path("/home/user/jiangjie/Jiangjie_Project")

RP50_DIR = PROJECT_ROOT / "data" / "ResearchProject_50"
XML_DIR = PROJECT_ROOT / "data" / "raw_wsi" / "Processed"
SOFT_LABEL_DIR = PROJECT_ROOT / "data" / "ResearchProject_soft_labels"

SOFT_LABEL_DIR.mkdir(parents=True, exist_ok=True)

TILE_SIZE = 224
TILE_AREA = TILE_SIZE * TILE_SIZE

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RP50_DIR:", RP50_DIR, RP50_DIR.exists())
print("XML_DIR:", XML_DIR, XML_DIR.exists())
print("SOFT_LABEL_DIR:", SOFT_LABEL_DIR, SOFT_LABEL_DIR.exists())
print("TILE_SIZE:", TILE_SIZE)
print("TILE_AREA:", TILE_AREA)

PROJECT_ROOT: /home/user/jiangjie/Jiangjie_Project
RP50_DIR: /home/user/jiangjie/Jiangjie_Project/data/ResearchProject_50 True
XML_DIR: /home/user/jiangjie/Jiangjie_Project/data/raw_wsi/Processed True
SOFT_LABEL_DIR: /home/user/jiangjie/Jiangjie_Project/data/ResearchProject_soft_labels True
TILE_SIZE: 224
TILE_AREA: 50176


In [2]:
label_columns = [
    "Irregular epithelial stratification",
    "Loss of polarity of basal cells",
    "Drop shaped rete ridges",
    "Premature keratinization in single cells",
    "Loss of epithelial cell cohesion",
    "Abnormal variation in nuclear size",
    "Abnormal variation in nuclear shape",
    "Abnormal variation in cell size",
    "Abnormal variation in cell shape",
    "Increased N:C ratio",
    "Increased number and size of nucleoli",
    "Hyperchromasia",
]

print("Number of selected labels:", len(label_columns))

for i, label in enumerate(label_columns):
    print(i, label)

Number of selected labels: 12
0 Irregular epithelial stratification
1 Loss of polarity of basal cells
2 Drop shaped rete ridges
3 Premature keratinization in single cells
4 Loss of epithelial cell cohesion
5 Abnormal variation in nuclear size
6 Abnormal variation in nuclear shape
7 Abnormal variation in cell size
8 Abnormal variation in cell shape
9 Increased N:C ratio
10 Increased number and size of nucleoli
11 Hyperchromasia


In [3]:
def fix_filepath(df):
    df = df.copy()
    df["filepath"] = df["filepath"].str.replace(
        "/scr/user/jiangjie/Jiangjie_Project",
        "/home/user/jiangjie/Jiangjie_Project",
        regex=False
    )
    return df


train_csv_path = RP50_DIR / "final_df_train.csv"
val_csv_path = RP50_DIR / "final_df_val.csv"
test_csv_path = RP50_DIR / "final_df_test.csv"

print("Train CSV exists:", train_csv_path.exists(), train_csv_path)
print("Val CSV exists:", val_csv_path.exists(), val_csv_path)
print("Test CSV exists:", test_csv_path.exists(), test_csv_path)

train_df = fix_filepath(pd.read_csv(train_csv_path))
val_df = fix_filepath(pd.read_csv(val_csv_path))
test_df = fix_filepath(pd.read_csv(test_csv_path))

print("\nShapes:")
print("train_df:", train_df.shape)
print("val_df:", val_df.shape)
print("test_df:", test_df.shape)

print("\nUnique WSIs:")
print("train:", train_df["slide_name"].nunique())
print("val:", val_df["slide_name"].nunique())
print("test:", test_df["slide_name"].nunique())

print("\nColumns:")
print(train_df.columns.tolist())

Train CSV exists: True /home/user/jiangjie/Jiangjie_Project/data/ResearchProject_50/final_df_train.csv
Val CSV exists: True /home/user/jiangjie/Jiangjie_Project/data/ResearchProject_50/final_df_val.csv
Test CSV exists: True /home/user/jiangjie/Jiangjie_Project/data/ResearchProject_50/final_df_test.csv

Shapes:
train_df: (146848, 21)
val_df: (198083, 21)
test_df: (314284, 21)

Unique WSIs:
train: 7
val: 6
test: 6

Columns:
['filename', 'slide_name', 'x', 'y', 'Irregular epithelial stratification', 'Loss of polarity of basal cells', 'Drop shaped rete ridges', 'Increased number of mitotic figures', 'Abnormally superficial mitotic figures', 'Premature keratinization in single cells', 'Keratin pearls within rete ridges', 'Loss of epithelial cell cohesion', 'Abnormal variation in nuclear size', 'Abnormal variation in nuclear shape', 'Abnormal variation in cell size', 'Abnormal variation in cell shape', 'Increased N:C ratio', 'Atypical mitotic figures', 'Increased number and size of nucleoli'

In [4]:
sample_slide = "217-DP-17"

sample_df = test_df[test_df["slide_name"] == sample_slide].copy().reset_index(drop=True)

print("Sample slide:", sample_slide)
print("sample_df shape:", sample_df.shape)

sample_img_path = Path(sample_df.loc[0, "filepath"])

print("Sample image path:", sample_img_path)
print("Exists:", sample_img_path.exists())

assert sample_img_path.exists(), "Sample image path does not exist."

img = Image.open(sample_img_path)
print("Image size:", img.size)

assert img.size == (TILE_SIZE, TILE_SIZE), f"Expected {(TILE_SIZE, TILE_SIZE)}, got {img.size}"

print("Tile size check passed.")

Sample slide: 217-DP-17
sample_df shape: (21782, 21)
Sample image path: /home/user/jiangjie/Jiangjie_Project/data/ResearchProject_50/217-DP-17/217-DP-17_tile_0000000.jpg
Exists: True
Image size: (224, 224)
Tile size check passed.


In [5]:
def split_annotation_name(annotation_name, selected_labels):
    """
    Split XML annotation name into one or multiple selected labels.

    Example:
    "A,B,C" -> ["A", "B", "C"] if A/B/C are selected labels.
    """
    if annotation_name is None:
        return []

    selected_labels_set = set(selected_labels)

    parts = [p.strip() for p in str(annotation_name).split(",")]

    matched_labels = [p for p in parts if p in selected_labels_set]

    return matched_labels


# Quick test
test_names = [
    "Loss of polarity of basal cells",
    "Abnormal variation in nuclear size,Abnormal variation in nuclear shape,Abnormal variation in cell size",
    "Atypical mitotic figures",
    None
]

for name in test_names:
    print("Original:", name)
    print("Parsed:", split_annotation_name(name, label_columns))
    print()

Original: Loss of polarity of basal cells
Parsed: ['Loss of polarity of basal cells']

Original: Abnormal variation in nuclear size,Abnormal variation in nuclear shape,Abnormal variation in cell size
Parsed: ['Abnormal variation in nuclear size', 'Abnormal variation in nuclear shape', 'Abnormal variation in cell size']

Original: Atypical mitotic figures
Parsed: []

Original: None
Parsed: []



In [6]:
def parse_xml_polygons(xml_path, selected_labels):
    """
    Parse XML annotation polygons.

    Returns:
    label_to_polygons:
        dict[label] = list of shapely Polygon

    annotation_summary_df:
        original XML annotation-level summary

    expanded_annotation_df:
        expanded label-level summary after splitting comma-separated labels
    """
    tree = ET.parse(xml_path)
    root = tree.getroot()

    selected_labels = list(selected_labels)
    label_to_polygons = {label: [] for label in selected_labels}

    all_annotation_rows = []
    expanded_rows = []

    for ann_idx, ann in enumerate(root.iter("annotation")):
        raw_label = ann.attrib.get("name", None)
        ann_type = ann.attrib.get("type", None)

        points = []

        for p in ann.iter("p"):
            if "x" in p.attrib and "y" in p.attrib:
                points.append((float(p.attrib["x"]), float(p.attrib["y"])))

        matched_labels = split_annotation_name(raw_label, selected_labels)

        all_annotation_rows.append({
            "annotation_index": ann_idx,
            "raw_label": raw_label,
            "type": ann_type,
            "num_points": len(points),
            "matched_selected_labels": matched_labels,
            "num_matched_selected_labels": len(matched_labels)
        })

        if ann_type != "polygon":
            continue

        if len(points) < 3:
            continue

        if len(matched_labels) == 0:
            continue

        poly = Polygon(points)

        if not poly.is_valid:
            poly = poly.buffer(0)

        if poly.is_empty:
            continue

        # If one XML polygon has multiple labels,
        # assign the same polygon to every matched selected label.
        for label in matched_labels:
            label_to_polygons[label].append(poly)

            expanded_rows.append({
                "annotation_index": ann_idx,
                "raw_label": raw_label,
                "expanded_label": label,
                "type": ann_type,
                "num_points": len(points),
                "polygon_area": poly.area
            })

    annotation_summary_df = pd.DataFrame(all_annotation_rows)
    expanded_annotation_df = pd.DataFrame(expanded_rows)

    return label_to_polygons, annotation_summary_df, expanded_annotation_df

In [7]:
def build_label_union_geometries(label_to_polygons, selected_labels):
    """
    Build one union annotation region for each selected label.

    For each label:
    A_c = union of all XML polygons assigned to this label.
    """
    label_to_geometry = {}

    for label in selected_labels:
        polygons = label_to_polygons.get(label, [])

        if len(polygons) == 0:
            label_to_geometry[label] = None

        elif len(polygons) == 1:
            geom = polygons[0]

            if not geom.is_valid:
                geom = geom.buffer(0)

            label_to_geometry[label] = geom

        else:
            geom = unary_union(polygons)

            if not geom.is_valid:
                geom = geom.buffer(0)

            label_to_geometry[label] = geom

    return label_to_geometry

In [8]:
def generate_soft_labels_for_slide_union(slide_df, label_to_geometry, selected_labels, tile_size=224):
    """
    Generate union-area soft labels for one slide.

    For each tile and each selected label:
    soft_label = Area(tile ∩ A_c) / Area(tile)

    where A_c is the union annotation region of all polygons assigned to that label.
    """
    soft_df = slide_df.copy()

    tile_area = tile_size * tile_size

    for label in selected_labels:
        soft_df[label] = 0.0

    xs = soft_df["x"].astype(float).values
    ys = soft_df["y"].astype(float).values

    for label in selected_labels:
        geom = label_to_geometry.get(label, None)

        if geom is None or geom.is_empty:
            continue

        minx, miny, maxx, maxy = geom.bounds

        candidate_mask = (
            (xs + tile_size >= minx) &
            (xs <= maxx) &
            (ys + tile_size >= miny) &
            (ys <= maxy)
        )

        candidate_indices = np.where(candidate_mask)[0]

        print(f"\nFeature: {label}")
        print("Candidate tiles:", len(candidate_indices))

        soft_values = np.zeros(len(soft_df), dtype=np.float32)

        for idx in tqdm(candidate_indices, desc=f"Computing union overlap: {label}"):
            x = xs[idx]
            y = ys[idx]

            tile_geom = box(x, y, x + tile_size, y + tile_size)

            if not tile_geom.intersects(geom):
                continue

            intersection_area = tile_geom.intersection(geom).area
            ratio = intersection_area / tile_area

            ratio = max(0.0, min(1.0, ratio))

            soft_values[idx] = ratio

        soft_df[label] = soft_values

        print(
            "Soft > 0:",
            int((soft_values > 0).sum()),
            "| Boundary 0<soft<0.5:",
            int(((soft_values > 0) & (soft_values < 0.5)).sum()),
            "| Soft>=0.5:",
            int((soft_values >= 0.5).sum()),
            "| Max:",
            float(soft_values.max())
        )

    return soft_df

In [9]:
def compare_soft_to_hard(hard_df, soft_df, selected_labels, threshold=0.5):
    """
    Compare generated soft labels with Kee hard labels using threshold 0.5.

    This is only a sanity check.
    Perfect matching is not required because Kee hard labels are threshold-based,
    while this soft-label method is area-based continuous overlap.
    """
    rows = []

    for label in selected_labels:
        hard = hard_df[label].astype(int).values
        soft = soft_df[label].astype(float).values
        generated_hard = (soft >= threshold).astype(int)

        fp_mask = (hard == 0) & (generated_hard == 1)
        fn_mask = (hard == 1) & (generated_hard == 0)

        rows.append({
            "Feature": label,
            "Hard Positive Count": int(hard.sum()),
            "Soft > 0 Count": int((soft > 0).sum()),
            "Boundary Count (0<soft<0.5)": int(((soft > 0) & (soft < threshold)).sum()),
            "Soft>=0.5 Positive Count": int(generated_hard.sum()),
            "Mismatch Count": int((hard != generated_hard).sum()),
            "False Positive vs Hard": int(fp_mask.sum()),
            "False Negative vs Hard": int(fn_mask.sum()),
            "Match Rate": float((hard == generated_hard).mean()),
            "Soft Max Value": float(soft.max())
        })

    return pd.DataFrame(rows)

In [10]:
def validate_soft_label_generation_for_slide(split_name, slide_name, split_df, selected_labels, tile_size=224):
    """
    Validate union-area soft label generation for one slide.
    """
    slide_df = split_df[split_df["slide_name"] == slide_name].copy().reset_index(drop=True)

    xml_path = XML_DIR / f"{slide_name}.xml"

    print("=" * 100)
    print("Split:", split_name)
    print("Slide:", slide_name)
    print("Number of tiles:", len(slide_df))
    print("XML path:", xml_path)
    print("XML exists:", xml_path.exists())

    if not xml_path.exists():
        raise FileNotFoundError(f"XML file not found for slide {slide_name}: {xml_path}")

    label_to_polygons, annotation_summary_df, expanded_annotation_df = parse_xml_polygons(
        xml_path,
        selected_labels=selected_labels
    )

    label_to_geometry = build_label_union_geometries(
        label_to_polygons,
        selected_labels=selected_labels
    )

    soft_df = generate_soft_labels_for_slide_union(
        slide_df=slide_df,
        label_to_geometry=label_to_geometry,
        selected_labels=selected_labels,
        tile_size=tile_size
    )

    check_df = compare_soft_to_hard(
        hard_df=slide_df,
        soft_df=soft_df,
        selected_labels=selected_labels,
        threshold=0.5
    )

    polygon_count_rows = []

    for label in selected_labels:
        geom = label_to_geometry.get(label, None)

        polygon_count_rows.append({
            "Feature": label,
            "XML Polygon Count": len(label_to_polygons[label]),
            "Union Geometry Exists": geom is not None and (not geom.is_empty),
            "Union Geometry Area": float(geom.area) if geom is not None and (not geom.is_empty) else 0.0
        })

    polygon_count_df = pd.DataFrame(polygon_count_rows)

    return check_df, soft_df, annotation_summary_df, expanded_annotation_df, polygon_count_df

In [11]:
sample_check_df, sample_soft_df, sample_annotation_summary_df, sample_expanded_annotation_df, sample_polygon_count_df = validate_soft_label_generation_for_slide(
    split_name="test",
    slide_name="217-DP-17",
    split_df=test_df,
    selected_labels=label_columns,
    tile_size=TILE_SIZE
)

print("\nOriginal XML annotation label counts:")
display(sample_annotation_summary_df["raw_label"].value_counts())

print("\nExpanded selected label counts:")
display(sample_expanded_annotation_df["expanded_label"].value_counts())

print("\nPolygon / union geometry summary:")
display(sample_polygon_count_df)

print("\n217-DP-17 soft-label sanity check:")
display(sample_check_df)

print("\nRows with mismatch > 0:")
display(sample_check_df[sample_check_df["Mismatch Count"] > 0])

Split: test
Slide: 217-DP-17
Number of tiles: 21782
XML path: /home/user/jiangjie/Jiangjie_Project/data/raw_wsi/Processed/217-DP-17.xml
XML exists: True

Feature: Loss of polarity of basal cells
Candidate tiles: 1350


Computing union overlap: Loss of polarity of basal cells: 100%|██████████| 1350/1350 [00:00<00:00, 13157.88it/s]


Soft > 0: 95 | Boundary 0<soft<0.5: 66 | Soft>=0.5: 29 | Max: 0.9992082118988037

Feature: Abnormal variation in nuclear shape
Candidate tiles: 433


Computing union overlap: Abnormal variation in nuclear shape: 100%|██████████| 433/433 [00:00<00:00, 11933.17it/s]


Soft > 0: 130 | Boundary 0<soft<0.5: 45 | Soft>=0.5: 85 | Max: 1.0

Feature: Abnormal variation in cell shape
Candidate tiles: 549


...

,Feature,XML Polygon Count,Union Geometry Exists,Union Geometry Area
0,Irregular epithelial stratification,0,False,0.000000e+00
1,Loss of polarity of basal cells,6,True,1.535820e+06
2,Drop shaped rete ridges,0,False,0.000000e+00
3,Premature keratinization in single cells,0,False,0.000000e+00
4,Loss of epithelial cell cohesion,0,False,0.000000e+00
5,Abnormal variation in nuclear size,0,False,0.000000e+00
6,Abnormal variation in nuclear shape,3,True,4.288468e+06
7,Abnormal variation in cell size,0,False,0.000000e+00
8,Abnormal variation in cell shape,3,True,8.390051e+06
9,Increased N:C ratio,0,False,0.000000e+00



217-DP-17 soft-label sanity check:


,Feature,Hard Positive Count,Soft > 0 Count,Boundary Count (0<soft<0.5),Soft>=0.5 Positive Count,Mismatch Count,False Positive vs Hard,False Negative vs Hard,Match Rate,Soft Max Value
0,Irregular epithelial stratification,0,0,0,0,0,0,0,1.0,0.000000
1,Loss of polarity of basal cells,29,95,66,29,0,0,0,1.0,0.999208
2,Drop shaped rete ridges,0,0,0,0,0,0,0,1.0,0.000000
3,Premature keratinization in single cells,0,0,0,0,0,0,0,1.0,0.000000
4,Loss of epithelial cell cohesion,0,0,0,0,0,0,0,1.0,0.000000
5,Abnormal variation in nuclear size,0,0,0,0,0,0,0,1.0,0.000000
6,Abnormal variation in nuclear shape,85,130,45,85,0,0,0,1.0,1.000000
7,Abnormal variation in cell size,0,0,0,0,0,0,0,1.0,0.000000
8,Abnormal variation in cell shape,168,223,55,168,0,0,0,1.0,1.000000
9,Increased N:C ratio,0,0,0,0,0,0,0,1.0,0.000000



Rows with mismatch > 0:


,Feature,Hard Positive Count,Soft > 0 Count,Boundary Count (0<soft<0.5),Soft>=0.5 Positive Count,Mismatch Count,False Positive vs Hard,False Negative vs Hard,Match Rate,Soft Max Value


In [12]:
d706_check_df, d706_soft_df, d706_annotation_summary_df, d706_expanded_annotation_df, d706_polygon_count_df = validate_soft_label_generation_for_slide(
    split_name="train",
    slide_name="D706-12-1-i",
    split_df=train_df,
    selected_labels=label_columns,
    tile_size=TILE_SIZE
)

print("\nOriginal XML annotation label counts:")
display(d706_annotation_summary_df["raw_label"].value_counts())

print("\nExpanded selected label counts:")
display(d706_expanded_annotation_df["expanded_label"].value_counts())

print("\nPolygon / union geometry summary:")
display(d706_polygon_count_df)

print("\nD706-12-1-i soft-label sanity check:")
display(d706_check_df)

print("\nRows with mismatch > 0:")
display(d706_check_df[d706_check_df["Mismatch Count"] > 0])

Split: train
Slide: D706-12-1-i
Number of tiles: 6763
XML path: /home/user/jiangjie/Jiangjie_Project/data/raw_wsi/Processed/D706-12-1-i.xml
XML exists: True

Feature: Irregular epithelial stratification
Candidate tiles: 1124


Computing union overlap: Irregular epithelial stratification: 100%|██████████| 1124/1124 [00:00<00:00, 18336.18it/s]


Soft > 0: 16 | Boundary 0<soft<0.5: 8 | Soft>=0.5: 8 | Max: 1.0

Feature: Loss of polarity of basal cells
Candidate tiles: 4364


Computing union overlap: Loss of polarity of basal cells: 100%|██████████| 4364/4364 [00:00<00:00, 22005.49it/s]


Soft > 0: 337 | Boundary 0<soft<0.5: 171 | Soft>=0.5: 166 | Max: 1.0

Feature: Abnormal variation in nuclear size
Candidate tiles: 5726


...

,Feature,XML Polygon Count,Union Geometry Exists,Union Geometry Area
0,Irregular epithelial stratification,2,True,3.688050e+05
1,Loss of polarity of basal cells,12,True,8.788188e+06
2,Drop shaped rete ridges,0,False,0.000000e+00
3,Premature keratinization in single cells,0,False,0.000000e+00
4,Loss of epithelial cell cohesion,0,False,0.000000e+00
5,Abnormal variation in nuclear size,13,True,7.177918e+07
6,Abnormal variation in nuclear shape,8,True,5.561361e+07
7,Abnormal variation in cell size,13,True,7.177918e+07
8,Abnormal variation in cell shape,8,True,5.561361e+07
9,Increased N:C ratio,13,True,7.177918e+07



D706-12-1-i soft-label sanity check:


,Feature,Hard Positive Count,Soft > 0 Count,Boundary Count (0<soft<0.5),Soft>=0.5 Positive Count,Mismatch Count,False Positive vs Hard,False Negative vs Hard,Match Rate,Soft Max Value
0,Irregular epithelial stratification,8,16,8,8,0,0,0,1.000000,1.0
1,Loss of polarity of basal cells,123,337,171,166,61,52,9,0.990980,1.0
2,Drop shaped rete ridges,0,0,0,0,0,0,0,1.000000,0.0
3,Premature keratinization in single cells,0,0,0,0,0,0,0,1.000000,0.0
4,Loss of epithelial cell cohesion,0,0,0,0,0,0,0,1.000000,0.0
5,Abnormal variation in nuclear size,1300,1681,318,1363,63,63,0,0.990685,1.0
6,Abnormal variation in nuclear shape,1019,1278,232,1046,27,27,0,0.996008,1.0
7,Abnormal variation in cell size,1300,1681,318,1363,63,63,0,0.990685,1.0
8,Abnormal variation in cell shape,1019,1278,232,1046,27,27,0,0.996008,1.0
9,Increased N:C ratio,1300,1681,318,1363,63,63,0,0.990685,1.0



Rows with mismatch > 0:


,Feature,Hard Positive Count,Soft > 0 Count,Boundary Count (0<soft<0.5),Soft>=0.5 Positive Count,Mismatch Count,False Positive vs Hard,False Negative vs Hard,Match Rate,Soft Max Value
1,Loss of polarity of basal cells,123,337,171,166,61,52,9,0.990980,1.0
5,Abnormal variation in nuclear size,1300,1681,318,1363,63,63,0,0.990685,1.0
6,Abnormal variation in nuclear shape,1019,1278,232,1046,27,27,0,0.996008,1.0
7,Abnormal variation in cell size,1300,1681,318,1363,63,63,0,0.990685,1.0
8,Abnormal variation in cell shape,1019,1278,232,1046,27,27,0,0.996008,1.0
9,Increased N:C ratio,1300,1681,318,1363,63,63,0,0.990685,1.0
10,Increased number and size of nucleoli,735,1004,206,798,63,63,0,0.990685,1.0


In [13]:
def generate_soft_labels_for_slide_union(slide_df, label_to_geometry, selected_labels, tile_size=224, verbose=False):
    """
    Generate union-area soft labels for one slide.

    For each tile and each selected label:
    soft_label = Area(tile ∩ A_c) / Area(tile)

    where A_c is the union annotation region of all polygons assigned to that label.
    """
    soft_df = slide_df.copy()
    tile_area = tile_size * tile_size

    for label in selected_labels:
        soft_df[label] = 0.0

    xs = soft_df["x"].astype(float).values
    ys = soft_df["y"].astype(float).values

    for label in selected_labels:
        geom = label_to_geometry.get(label, None)

        if geom is None or geom.is_empty:
            continue

        minx, miny, maxx, maxy = geom.bounds

        candidate_mask = (
            (xs + tile_size >= minx) &
            (xs <= maxx) &
            (ys + tile_size >= miny) &
            (ys <= maxy)
        )

        candidate_indices = np.where(candidate_mask)[0]

        if verbose:
            print("\nFeature:", label)
            print("Candidate tiles:", len(candidate_indices))

        soft_values = np.zeros(len(soft_df), dtype=np.float32)

        iterator = tqdm(
            candidate_indices,
            desc=f"Computing union overlap: {label}",
            disable=not verbose
        )

        for idx in iterator:
            x = xs[idx]
            y = ys[idx]

            tile_geom = box(x, y, x + tile_size, y + tile_size)

            if not tile_geom.intersects(geom):
                continue

            intersection_area = tile_geom.intersection(geom).area
            ratio = intersection_area / tile_area
            ratio = max(0.0, min(1.0, ratio))

            soft_values[idx] = ratio

        soft_df[label] = soft_values

        if verbose:
            print(
                "Soft > 0:",
                int((soft_values > 0).sum()),
                "| Boundary 0<soft<0.5:",
                int(((soft_values > 0) & (soft_values < 0.5)).sum()),
                "| Soft>=0.5:",
                int((soft_values >= 0.5).sum()),
                "| Max:",
                float(soft_values.max())
            )

    return soft_df

In [14]:
def validate_soft_label_generation_for_slide_quiet(split_name, slide_name, split_df, selected_labels, tile_size=224):
    """
    Quiet validation for one slide.
    Used for multi-slide sanity check.
    """
    slide_df = split_df[split_df["slide_name"] == slide_name].copy().reset_index(drop=True)

    xml_path = XML_DIR / f"{slide_name}.xml"

    if len(slide_df) == 0:
        raise ValueError(f"No tiles found for slide {slide_name} in {split_name}")

    if not xml_path.exists():
        raise FileNotFoundError(f"XML file not found for slide {slide_name}: {xml_path}")

    label_to_polygons, annotation_summary_df, expanded_annotation_df = parse_xml_polygons(
        xml_path,
        selected_labels=selected_labels
    )

    label_to_geometry = build_label_union_geometries(
        label_to_polygons,
        selected_labels=selected_labels
    )

    soft_df = generate_soft_labels_for_slide_union(
        slide_df=slide_df,
        label_to_geometry=label_to_geometry,
        selected_labels=selected_labels,
        tile_size=tile_size,
        verbose=False
    )

    check_df = compare_soft_to_hard(
        hard_df=slide_df,
        soft_df=soft_df,
        selected_labels=selected_labels,
        threshold=0.5
    )

    check_df.insert(0, "Split", split_name)
    check_df.insert(1, "Slide", slide_name)
    check_df.insert(2, "NumTiles", len(slide_df))

    polygon_rows = []

    for label in selected_labels:
        geom = label_to_geometry.get(label, None)

        polygon_rows.append({
            "Split": split_name,
            "Slide": slide_name,
            "Feature": label,
            "NumTiles": len(slide_df),
            "XML Polygon Count": len(label_to_polygons[label]),
            "Union Geometry Exists": geom is not None and (not geom.is_empty),
            "Union Geometry Area": float(geom.area) if geom is not None and (not geom.is_empty) else 0.0
        })

    polygon_count_df = pd.DataFrame(polygon_rows)

    annotation_label_counts = annotation_summary_df["raw_label"].value_counts().reset_index()
    annotation_label_counts.columns = ["raw_label", "count"]
    annotation_label_counts.insert(0, "Split", split_name)
    annotation_label_counts.insert(1, "Slide", slide_name)

    expanded_label_counts = expanded_annotation_df["expanded_label"].value_counts().reset_index()
    expanded_label_counts.columns = ["expanded_label", "count"]
    expanded_label_counts.insert(0, "Split", split_name)
    expanded_label_counts.insert(1, "Slide", slide_name)

    return check_df, polygon_count_df, annotation_label_counts, expanded_label_counts

In [15]:
sanity_slides = {
    "train": ["D706-12-1-i", "707-DP-17-2"],
    "validation": ["1467-DP-12", "D706-12-1-i"],
    "test": ["217-DP-17", "1549-DP-17", "807-DP-17"]
}

split_to_df = {
    "train": train_df,
    "validation": val_df,
    "test": test_df
}

available_sanity_slides = {}

for split_name, slide_list in sanity_slides.items():
    split_df = split_to_df[split_name]
    available_slides = []

    print("\nSplit:", split_name)

    for slide in slide_list:
        exists_in_csv = slide in set(split_df["slide_name"].unique())
        xml_exists = (XML_DIR / f"{slide}.xml").exists()

        print(slide, "| in CSV:", exists_in_csv, "| XML:", xml_exists)

        if exists_in_csv and xml_exists:
            available_slides.append(slide)

    available_sanity_slides[split_name] = available_slides

print("\nAvailable sanity slides:")
print(available_sanity_slides)


Split: train
D706-12-1-i | in CSV: True | XML: True
707-DP-17-2 | in CSV: True | XML: True

Split: validation
1467-DP-12 | in CSV: True | XML: True
D706-12-1-i | in CSV: True | XML: True

Split: test
217-DP-17 | in CSV: True | XML: True
1549-DP-17 | in CSV: True | XML: True
807-DP-17 | in CSV: True | XML: True

Available sanity slides:
{'train': ['D706-12-1-i', '707-DP-17-2'], 'validation': ['1467-DP-12', 'D706-12-1-i'], 'test': ['217-DP-17', '1549-DP-17', '807-DP-17']}


In [16]:
all_check_dfs = []
all_polygon_dfs = []
all_annotation_label_count_dfs = []
all_expanded_label_count_dfs = []

for split_name, slide_list in available_sanity_slides.items():
    split_df = split_to_df[split_name]

    for slide_name in slide_list:
        print("=" * 100)
        print("Running sanity check:", split_name, slide_name)

        check_df, polygon_count_df, annotation_label_counts, expanded_label_counts = validate_soft_label_generation_for_slide_quiet(
            split_name=split_name,
            slide_name=slide_name,
            split_df=split_df,
            selected_labels=label_columns,
            tile_size=TILE_SIZE
        )

        all_check_dfs.append(check_df)
        all_polygon_dfs.append(polygon_count_df)
        all_annotation_label_count_dfs.append(annotation_label_counts)
        all_expanded_label_count_dfs.append(expanded_label_counts)

sanity_check_df = pd.concat(all_check_dfs, ignore_index=True)
sanity_polygon_df = pd.concat(all_polygon_dfs, ignore_index=True)
sanity_annotation_label_counts_df = pd.concat(all_annotation_label_count_dfs, ignore_index=True)
sanity_expanded_label_counts_df = pd.concat(all_expanded_label_count_dfs, ignore_index=True)

print("sanity_check_df shape:", sanity_check_df.shape)
print("sanity_polygon_df shape:", sanity_polygon_df.shape)
print("sanity_annotation_label_counts_df shape:", sanity_annotation_label_counts_df.shape)
print("sanity_expanded_label_counts_df shape:", sanity_expanded_label_counts_df.shape)

Running sanity check: train D706-12-1-i
Running sanity check: train 707-DP-17-2
Running sanity check: validation 1467-DP-12
Running sanity check: validation D706-12-1-i
Running sanity check: test 217-DP-17
Running sanity check: test 1549-DP-17
Running sanity check: test 807-DP-17
sanity_check_df shape: (84, 13)
sanity_polygon_df shape: (84, 7)
sanity_annotation_label_counts_df shape: (60, 4)
sanity_expanded_label_counts_df shape: (63, 4)


In [17]:
summary_rows = []

for (split_name, slide_name), group in sanity_check_df.groupby(["Split", "Slide"]):
    num_tiles = int(group["NumTiles"].iloc[0])
    total_comparisons = num_tiles * len(label_columns)
    total_mismatch = int(group["Mismatch Count"].sum())

    summary_rows.append({
        "Split": split_name,
        "Slide": slide_name,
        "NumTiles": num_tiles,
        "Total Hard Positive Count": int(group["Hard Positive Count"].sum()),
        "Total Soft > 0 Count": int(group["Soft > 0 Count"].sum()),
        "Total Boundary Count (0<soft<0.5)": int(group["Boundary Count (0<soft<0.5)"].sum()),
        "Total Soft>=0.5 Positive Count": int(group["Soft>=0.5 Positive Count"].sum()),
        "Total Mismatch Count": total_mismatch,
        "Total False Positive vs Hard": int(group["False Positive vs Hard"].sum()),
        "Total False Negative vs Hard": int(group["False Negative vs Hard"].sum()),
        "Overall Match Rate": 1 - (total_mismatch / total_comparisons),
        "Min Feature Match Rate": float(group["Match Rate"].min()),
        "Max Soft Value": float(group["Soft Max Value"].max())
    })

sanity_summary_df = pd.DataFrame(summary_rows)

display(sanity_summary_df.sort_values(["Split", "Slide"]))

,Split,Slide,NumTiles,Total Hard Positive Count,Total Soft > 0 Count,Total Boundary Count (0<soft<0.5),Total Soft>=0.5 Positive Count,Total Mismatch Count,Total False Positive vs Hard,Total False Negative vs Hard,Overall Match Rate,Min Feature Match Rate,Max Soft Value
0,test,1549-DP-17,66670,6927,8443,1505,6938,11,11,0,0.999986,0.999925,1.0
1,test,217-DP-17,21782,282,448,166,282,0,0,0,1.000000,1.000000,1.0
2,test,807-DP-17,19541,2508,2970,460,2510,2,2,0,0.999991,0.999898,1.0
3,train,707-DP-17-2,40834,5414,6210,793,5417,3,3,0,0.999994,0.999951,1.0
4,train,D706-12-1-i,6763,7369,9633,1915,7718,367,358,9,0.995478,0.990685,1.0
5,validation,1467-DP-12,30600,965,1511,545,966,1,1,0,0.999997,0.999967,1.0
6,validation,D706-12-1-i,6763,7369,9633,1915,7718,367,358,9,0.995478,0.990685,1.0


In [18]:
print("Rows with mismatch > 0, sorted by mismatch count:")

mismatch_rows_df = sanity_check_df[
    sanity_check_df["Mismatch Count"] > 0
].sort_values(
    ["Mismatch Count", "Split", "Slide"],
    ascending=[False, True, True]
)

display(mismatch_rows_df)

Rows with mismatch > 0, sorted by mismatch count:


,Split,Slide,NumTiles,Feature,Hard Positive Count,Soft > 0 Count,Boundary Count (0<soft<0.5),Soft>=0.5 Positive Count,Mismatch Count,False Positive vs Hard,False Negative vs Hard,Match Rate,Soft Max Value
5,train,D706-12-1-i,6763,Abnormal variation in nuclear size,1300,1681,318,1363,63,63,0,0.990685,1.0
7,train,D706-12-1-i,6763,Abnormal variation in cell size,1300,1681,318,1363,63,63,0,0.990685,1.0
9,train,D706-12-1-i,6763,Increased N:C ratio,1300,1681,318,1363,63,63,0,0.990685,1.0
10,train,D706-12-1-i,6763,Increased number and size of nucleoli,735,1004,206,798,63,63,0,0.990685,1.0
41,validation,D706-12-1-i,6763,Abnormal variation in nuclear size,1300,1681,318,1363,63,63,0,0.990685,1.0
43,validation,D706-12-1-i,6763,Abnormal variation in cell size,1300,1681,318,1363,63,63,0,0.990685,1.0
45,validation,D706-12-1-i,6763,Increased N:C ratio,1300,1681,318,1363,63,63,0,0.990685,1.0
46,validation,D706-12-1-i,6763,Increased number and size of nucleoli,735,1004,206,798,63,63,0,0.990685,1.0
1,train,D706-12-1-i,6763,Loss of polarity of basal cells,123,337,171,166,61,52,9,0.990980,1.0
37,validation,D706-12-1-i,6763,Loss of polarity of basal cells,123,337,171,166,61,52,9,0.990980,1.0


In [19]:
print("Rows sorted by boundary count:")

boundary_rows_df = sanity_check_df.sort_values(
    "Boundary Count (0<soft<0.5)",
    ascending=False
)

display(boundary_rows_df.head(30))

Rows sorted by boundary count:


,Split,Slide,NumTiles,Feature,Hard Positive Count,Soft > 0 Count,Boundary Count (0<soft<0.5),Soft>=0.5 Positive Count,Mismatch Count,False Positive vs Hard,False Negative vs Hard,Match Rate,Soft Max Value
63,test,1549-DP-17,66670,Premature keratinization in single cells,2939,3388,448,2940,1,1,0,0.999985,1.000000
7,train,D706-12-1-i,6763,Abnormal variation in cell size,1300,1681,318,1363,63,63,0,0.990685,1.000000
5,train,D706-12-1-i,6763,Abnormal variation in nuclear size,1300,1681,318,1363,63,63,0,0.990685,1.000000
9,train,D706-12-1-i,6763,Increased N:C ratio,1300,1681,318,1363,63,63,0,0.990685,1.000000
45,validation,D706-12-1-i,6763,Increased N:C ratio,1300,1681,318,1363,63,63,0,0.990685,1.000000
43,validation,D706-12-1-i,6763,Abnormal variation in cell size,1300,1681,318,1363,63,63,0,0.990685,1.000000
41,validation,D706-12-1-i,6763,Abnormal variation in nuclear size,1300,1681,318,1363,63,63,0,0.990685,1.000000
44,validation,D706-12-1-i,6763,Abnormal variation in cell shape,1019,1278,232,1046,27,27,0,0.996008,1.000000
8,train,D706-12-1-i,6763,Abnormal variation in cell shape,1019,1278,232,1046,27,27,0,0.996008,1.000000
6,train,D706-12-1-i,6763,Abnormal variation in nuclear shape,1019,1278,232,1046,27,27,0,0.996008,1.000000


In [20]:
sanity_check_path = SOFT_LABEL_DIR / "soft_label_multislide_sanity_check_union_corrected.csv"
sanity_summary_path = SOFT_LABEL_DIR / "soft_label_multislide_sanity_summary_union_corrected.csv"
sanity_polygon_path = SOFT_LABEL_DIR / "soft_label_multislide_polygon_summary_union_corrected.csv"
sanity_annotation_counts_path = SOFT_LABEL_DIR / "soft_label_multislide_raw_annotation_counts_union_corrected.csv"
sanity_expanded_counts_path = SOFT_LABEL_DIR / "soft_label_multislide_expanded_label_counts_union_corrected.csv"

sanity_check_df.to_csv(sanity_check_path, index=False)
sanity_summary_df.to_csv(sanity_summary_path, index=False)
sanity_polygon_df.to_csv(sanity_polygon_path, index=False)
sanity_annotation_label_counts_df.to_csv(sanity_annotation_counts_path, index=False)
sanity_expanded_label_counts_df.to_csv(sanity_expanded_counts_path, index=False)

print("Saved:")
print(sanity_check_path)
print(sanity_summary_path)
print(sanity_polygon_path)
print(sanity_annotation_counts_path)
print(sanity_expanded_counts_path)

Saved:
/home/user/jiangjie/Jiangjie_Project/data/ResearchProject_soft_labels/soft_label_multislide_sanity_check_union_corrected.csv
/home/user/jiangjie/Jiangjie_Project/data/ResearchProject_soft_labels/soft_label_multislide_sanity_summary_union_corrected.csv
/home/user/jiangjie/Jiangjie_Project/data/ResearchProject_soft_labels/soft_label_multislide_polygon_summary_union_corrected.csv
/home/user/jiangjie/Jiangjie_Project/data/ResearchProject_soft_labels/soft_label_multislide_raw_annotation_counts_union_corrected.csv
/home/user/jiangjie/Jiangjie_Project/data/ResearchProject_soft_labels/soft_label_multislide_expanded_label_counts_union_corrected.csv


In [ ]:
本研究在 train、validation 和 test splits 中选取 7 张代表性 WSI 进行 multi-slide sanity check。
结果显示，修正后的 union-area soft-label generation 方法在以 0.5 threshold 重新二值化后，与 Kee hard labels 具有高度一致性。
其中 5 张 slides 的 mismatch 数量仅为 0–11。
唯一 mismatch 相对较多的是 D706-12-1-i，该 slide 包含复杂的 comma-separated multi-label XML annotations 以及同一 feature 的多个 polygons。
即使如此，该 slide 的 overall match rate 仍达到约 99.55%。因此，该结果说明 corrected XML parser 与 union-area overlap calculation 是稳定且可靠的。
剩余 mismatch 主要反映 Kee threshold-based hard-label rule 与本研究 area-based continuous soft-label reconstruction 之间的定义差异。

In [21]:
def summarize_soft_value_bins_for_slide(soft_df, split_name, slide_name, selected_labels):
    """
    Summarize soft label value distribution for one slide.

    Bins:
    exactly 0
    0 < soft < 0.1
    0.1 <= soft < 0.2
    0.2 <= soft < 0.3
    0.3 <= soft < 0.4
    0.4 <= soft < 0.5
    0.5 <= soft <= 1.0
    """
    rows = []

    for label in selected_labels:
        values = soft_df[label].astype(float).values

        total_tiles = len(values)
        zero_count = int((values == 0).sum())
        nonzero_count = int((values > 0).sum())

        bin_0_01 = int(((values > 0.0) & (values < 0.1)).sum())
        bin_01_02 = int(((values >= 0.1) & (values < 0.2)).sum())
        bin_02_03 = int(((values >= 0.2) & (values < 0.3)).sum())
        bin_03_04 = int(((values >= 0.3) & (values < 0.4)).sum())
        bin_04_05 = int(((values >= 0.4) & (values < 0.5)).sum())
        bin_05_10 = int(((values >= 0.5) & (values <= 1.0)).sum())

        boundary_count = bin_0_01 + bin_01_02 + bin_02_03 + bin_03_04 + bin_04_05
        informative_boundary_count = bin_03_04 + bin_04_05

        rows.append({
            "Split": split_name,
            "Slide": slide_name,
            "Feature": label,
            "Total Tiles": total_tiles,
            "Soft = 0 Count": zero_count,
            "Soft > 0 Count": nonzero_count,
            "0 < soft < 0.1": bin_0_01,
            "0.1 <= soft < 0.2": bin_01_02,
            "0.2 <= soft < 0.3": bin_02_03,
            "0.3 <= soft < 0.4": bin_03_04,
            "0.4 <= soft < 0.5": bin_04_05,
            "0.5 <= soft <= 1.0": bin_05_10,
            "Boundary Count (0<soft<0.5)": boundary_count,
            "Informative Boundary Count (0.3<=soft<0.5)": informative_boundary_count,
            "Tiny Boundary Count (0<soft<0.1)": bin_0_01,
            "Informative Boundary Ratio among Boundary": (
                informative_boundary_count / boundary_count if boundary_count > 0 else 0.0
            ),
            "Tiny Boundary Ratio among Boundary": (
                bin_0_01 / boundary_count if boundary_count > 0 else 0.0
            ),
        })

    return pd.DataFrame(rows)

In [22]:
all_bin_distribution_dfs = []

for split_name, slide_list in available_sanity_slides.items():
    split_df = split_to_df[split_name]

    for slide_name in slide_list:
        print("=" * 100)
        print("Computing soft-value bin distribution:", split_name, slide_name)

        slide_df = split_df[split_df["slide_name"] == slide_name].copy().reset_index(drop=True)

        xml_path = XML_DIR / f"{slide_name}.xml"

        label_to_polygons, annotation_summary_df, expanded_annotation_df = parse_xml_polygons(
            xml_path,
            selected_labels=label_columns
        )

        label_to_geometry = build_label_union_geometries(
            label_to_polygons,
            selected_labels=label_columns
        )

        soft_df = generate_soft_labels_for_slide_union(
            slide_df=slide_df,
            label_to_geometry=label_to_geometry,
            selected_labels=label_columns,
            tile_size=TILE_SIZE,
            verbose=False
        )

        bin_df = summarize_soft_value_bins_for_slide(
            soft_df=soft_df,
            split_name=split_name,
            slide_name=slide_name,
            selected_labels=label_columns
        )

        all_bin_distribution_dfs.append(bin_df)

soft_value_bin_df = pd.concat(all_bin_distribution_dfs, ignore_index=True)

print("soft_value_bin_df shape:", soft_value_bin_df.shape)
display(soft_value_bin_df.head())

Computing soft-value bin distribution: train D706-12-1-i
Computing soft-value bin distribution: train 707-DP-17-2
Computing soft-value bin distribution: validation 1467-DP-12
Computing soft-value bin distribution: validation D706-12-1-i
Computing soft-value bin distribution: test 217-DP-17
Computing soft-value bin distribution: test 1549-DP-17
Computing soft-value bin distribution: test 807-DP-17
soft_value_bin_df shape: (84, 17)


,Split,Slide,Feature,Total Tiles,Soft = 0 Count,Soft > 0 Count,0 < soft < 0.1,0.1 <= soft < 0.2,0.2 <= soft < 0.3,0.3 <= soft < 0.4,0.4 <= soft < 0.5,0.5 <= soft <= 1.0,Boundary Count (0<soft<0.5),Informative Boundary Count (0.3<=soft<0.5),Tiny Boundary Count (0<soft<0.1),Informative Boundary Ratio among Boundary,Tiny Boundary Ratio among Boundary
0,train,D706-12-1-i,Irregular epithelial stratification,6763,6747,16,3,0,3,2,0,8,8,2,3,0.250000,0.375000
1,train,D706-12-1-i,Loss of polarity of basal cells,6763,6426,337,80,27,21,21,22,166,171,43,80,0.251462,0.467836
2,train,D706-12-1-i,Drop shaped rete ridges,6763,6763,0,0,0,0,0,0,0,0,0,0,0.000000,0.000000
3,train,D706-12-1-i,Premature keratinization in single cells,6763,6763,0,0,0,0,0,0,0,0,0,0,0.000000,0.000000
4,train,D706-12-1-i,Loss of epithelial cell cohesion,6763,6763,0,0,0,0,0,0,0,0,0,0,0.000000,0.000000


In [23]:
bin_columns = [
    "Soft = 0 Count",
    "Soft > 0 Count",
    "0 < soft < 0.1",
    "0.1 <= soft < 0.2",
    "0.2 <= soft < 0.3",
    "0.3 <= soft < 0.4",
    "0.4 <= soft < 0.5",
    "0.5 <= soft <= 1.0",
    "Boundary Count (0<soft<0.5)",
    "Informative Boundary Count (0.3<=soft<0.5)",
    "Tiny Boundary Count (0<soft<0.1)"
]

overall_bin_summary = soft_value_bin_df[bin_columns].sum().to_frame(name="Count")

display(overall_bin_summary)

,Count
Soft = 0 Count,2276588
Soft > 0 Count,38848
0 < soft < 0.1,3121
0.1 <= soft < 0.2,1285
0.2 <= soft < 0.3,1054
0.3 <= soft < 0.4,920
0.4 <= soft < 0.5,919
0.5 <= soft <= 1.0,31549
Boundary Count (0<soft<0.5),7299
Informative Boundary Count (0.3<=soft<0.5),1839


In [24]:
split_bin_summary = soft_value_bin_df.groupby("Split")[bin_columns].sum().reset_index()

display(split_bin_summary)

,Split,Soft = 0 Count,Soft > 0 Count,0 < soft < 0.1,0.1 <= soft < 0.2,0.2 <= soft < 0.3,0.3 <= soft < 0.4,0.4 <= soft < 0.5,0.5 <= soft <= 1.0,Boundary Count (0<soft<0.5),Informative Boundary Count (0.3<=soft<0.5),Tiny Boundary Count (0<soft<0.1)
0,test,1284055,11861,891,378,312,282,268,9730,2131,550,891
1,train,555321,15843,1164,472,381,344,347,13135,2708,691,1164
2,validation,437212,11144,1066,435,361,294,304,8684,2460,598,1066


In [25]:
informative_boundary_rows = soft_value_bin_df.sort_values(
    "Informative Boundary Count (0.3<=soft<0.5)",
    ascending=False
)

display(informative_boundary_rows.head(30))

,Split,Slide,Feature,Total Tiles,Soft = 0 Count,Soft > 0 Count,0 < soft < 0.1,0.1 <= soft < 0.2,0.2 <= soft < 0.3,0.3 <= soft < 0.4,0.4 <= soft < 0.5,0.5 <= soft <= 1.0,Boundary Count (0<soft<0.5),Informative Boundary Count (0.3<=soft<0.5),Tiny Boundary Count (0<soft<0.1),Informative Boundary Ratio among Boundary,Tiny Boundary Ratio among Boundary
63,test,1549-DP-17,Premature keratinization in single cells,66670,63282,3388,201,79,59,57,52,2940,448,109,201,0.243304,0.448661
7,train,D706-12-1-i,Abnormal variation in cell size,6763,5082,1681,143,54,47,37,37,1363,318,74,143,0.232704,0.449686
5,train,D706-12-1-i,Abnormal variation in nuclear size,6763,5082,1681,143,54,47,37,37,1363,318,74,143,0.232704,0.449686
9,train,D706-12-1-i,Increased N:C ratio,6763,5082,1681,143,54,47,37,37,1363,318,74,143,0.232704,0.449686
45,validation,D706-12-1-i,Increased N:C ratio,6763,5082,1681,143,54,47,37,37,1363,318,74,143,0.232704,0.449686
43,validation,D706-12-1-i,Abnormal variation in cell size,6763,5082,1681,143,54,47,37,37,1363,318,74,143,0.232704,0.449686
41,validation,D706-12-1-i,Abnormal variation in nuclear size,6763,5082,1681,143,54,47,37,37,1363,318,74,143,0.232704,0.449686
22,train,707-DP-17-2,Increased number and size of nucleoli,40834,38956,1878,56,30,31,31,41,1689,189,72,56,0.380952,0.296296
68,test,1549-DP-17,Abnormal variation in cell shape,66670,65756,914,94,40,30,22,39,689,225,61,94,0.271111,0.417778
66,test,1549-DP-17,Abnormal variation in nuclear shape,66670,65781,889,92,39,31,20,38,669,220,58,92,0.263636,0.418182


In [26]:
tiny_boundary_rows = soft_value_bin_df.sort_values(
    "Tiny Boundary Count (0<soft<0.1)",
    ascending=False
)

display(tiny_boundary_rows.head(30))

,Split,Slide,Feature,Total Tiles,Soft = 0 Count,Soft > 0 Count,0 < soft < 0.1,0.1 <= soft < 0.2,0.2 <= soft < 0.3,0.3 <= soft < 0.4,0.4 <= soft < 0.5,0.5 <= soft <= 1.0,Boundary Count (0<soft<0.5),Informative Boundary Count (0.3<=soft<0.5),Tiny Boundary Count (0<soft<0.1),Informative Boundary Ratio among Boundary,Tiny Boundary Ratio among Boundary
63,test,1549-DP-17,Premature keratinization in single cells,66670,63282,3388,201,79,59,57,52,2940,448,109,201,0.243304,0.448661
7,train,D706-12-1-i,Abnormal variation in cell size,6763,5082,1681,143,54,47,37,37,1363,318,74,143,0.232704,0.449686
5,train,D706-12-1-i,Abnormal variation in nuclear size,6763,5082,1681,143,54,47,37,37,1363,318,74,143,0.232704,0.449686
9,train,D706-12-1-i,Increased N:C ratio,6763,5082,1681,143,54,47,37,37,1363,318,74,143,0.232704,0.449686
45,validation,D706-12-1-i,Increased N:C ratio,6763,5082,1681,143,54,47,37,37,1363,318,74,143,0.232704,0.449686
43,validation,D706-12-1-i,Abnormal variation in cell size,6763,5082,1681,143,54,47,37,37,1363,318,74,143,0.232704,0.449686
41,validation,D706-12-1-i,Abnormal variation in nuclear size,6763,5082,1681,143,54,47,37,37,1363,318,74,143,0.232704,0.449686
44,validation,D706-12-1-i,Abnormal variation in cell shape,6763,5485,1278,110,42,32,20,28,1046,232,48,110,0.206897,0.474138
8,train,D706-12-1-i,Abnormal variation in cell shape,6763,5485,1278,110,42,32,20,28,1046,232,48,110,0.206897,0.474138
6,train,D706-12-1-i,Abnormal variation in nuclear shape,6763,5485,1278,110,42,32,20,28,1046,232,48,110,0.206897,0.474138


In [27]:
soft_value_bin_path = SOFT_LABEL_DIR / "soft_value_bin_distribution_sanity_slides_union_corrected.csv"
overall_bin_summary_path = SOFT_LABEL_DIR / "soft_value_bin_distribution_overall_summary_union_corrected.csv"
split_bin_summary_path = SOFT_LABEL_DIR / "soft_value_bin_distribution_split_summary_union_corrected.csv"

soft_value_bin_df.to_csv(soft_value_bin_path, index=False)
overall_bin_summary.to_csv(overall_bin_summary_path)
split_bin_summary.to_csv(split_bin_summary_path, index=False)

print("Saved:")
print(soft_value_bin_path)
print(overall_bin_summary_path)
print(split_bin_summary_path)

Saved:
/home/user/jiangjie/Jiangjie_Project/data/ResearchProject_soft_labels/soft_value_bin_distribution_sanity_slides_union_corrected.csv
/home/user/jiangjie/Jiangjie_Project/data/ResearchProject_soft_labels/soft_value_bin_distribution_overall_summary_union_corrected.csv
/home/user/jiangjie/Jiangjie_Project/data/ResearchProject_soft_labels/soft_value_bin_distribution_split_summary_union_corrected.csv


In [30]:
# =========================
# Cell 29: Check source CSV files before full soft-label generation
# =========================

required_columns = ["filename", "slide_name", "x", "y"] + label_columns + ["filepath"]

source_csv_rows = []

candidate_files = []

# Train split files
for i in range(1, 11):
    candidate_files.append((f"train{i}", RP50_DIR / f"final_df_train{i}.csv"))

# Existing merged train, validation, test
candidate_files.append(("train", RP50_DIR / "final_df_train.csv"))
candidate_files.append(("validation", RP50_DIR / "final_df_val.csv"))
candidate_files.append(("test", RP50_DIR / "final_df_test.csv"))

for split_name, csv_path in candidate_files:
    exists = csv_path.exists()

    row = {
        "Split": split_name,
        "CSV Path": str(csv_path),
        "Exists": exists,
        "Num Rows": None,
        "Num Slides": None,
        "Columns OK": None,
        "Missing Required Columns": None,
    }

    if exists:
        # Read only header first
        header_df = pd.read_csv(csv_path, nrows=0)
        columns = header_df.columns.tolist()

        missing_columns = [col for col in required_columns if col not in columns]

        row["Columns OK"] = len(missing_columns) == 0
        row["Missing Required Columns"] = missing_columns

        # Read only slide_name to count rows/slides
        slide_df = pd.read_csv(csv_path, usecols=["slide_name"])
        row["Num Rows"] = len(slide_df)
        row["Num Slides"] = slide_df["slide_name"].nunique()

    source_csv_rows.append(row)

source_csv_check_df = pd.DataFrame(source_csv_rows)

display(source_csv_check_df[[
    "Split",
    "Exists",
    "Num Rows",
    "Num Slides",
    "Columns OK",
    "Missing Required Columns",
    "CSV Path"
]])

,Split,Exists,Num Rows,Num Slides,Columns OK,Missing Required Columns,CSV Path
0,train1,True,135946,7,True,[],/home/user/jiangjie/Jiangjie_Project/data/Rese...
1,train2,True,135946,3,True,[],/home/user/jiangjie/Jiangjie_Project/data/Rese...
2,train3,True,135946,3,True,[],/home/user/jiangjie/Jiangjie_Project/data/Rese...
3,train4,True,135946,1,True,[],/home/user/jiangjie/Jiangjie_Project/data/Rese...
4,train5,True,135946,1,True,[],/home/user/jiangjie/Jiangjie_Project/data/Rese...
5,train6,True,135946,1,True,[],/home/user/jiangjie/Jiangjie_Project/data/Rese...
6,train7,True,135946,1,True,[],/home/user/jiangjie/Jiangjie_Project/data/Rese...
7,train8,True,135946,5,True,[],/home/user/jiangjie/Jiangjie_Project/data/Rese...
8,train9,True,135946,3,True,[],/home/user/jiangjie/Jiangjie_Project/data/Rese...
9,train10,True,135946,3,True,[],/home/user/jiangjie/Jiangjie_Project/data/Rese...


In [31]:
# =========================
# Cell 30: Summarize train split files
# =========================

train_split_check_df = source_csv_check_df[
    source_csv_check_df["Split"].str.startswith("train")
].copy()

display(train_split_check_df[[
    "Split",
    "Exists",
    "Num Rows",
    "Num Slides",
    "Columns OK"
]])

train_part_df = source_csv_check_df[
    source_csv_check_df["Split"].isin([f"train{i}" for i in range(1, 11)])
].copy()

num_existing_train_parts = int(train_part_df["Exists"].sum())
total_train_part_rows = int(train_part_df["Num Rows"].fillna(0).sum())
total_train_part_slide_counts = int(train_part_df["Num Slides"].fillna(0).sum())

print("Number of existing train part files:", num_existing_train_parts)
print("Total rows across train1-train10:", total_train_part_rows)
print("Sum of slide counts across train1-train10:", total_train_part_slide_counts)

merged_train_row = source_csv_check_df[source_csv_check_df["Split"] == "train"]

if len(merged_train_row) > 0 and bool(merged_train_row["Exists"].iloc[0]):
    print("Rows in final_df_train.csv:", int(merged_train_row["Num Rows"].iloc[0]))
    print("Slides in final_df_train.csv:", int(merged_train_row["Num Slides"].iloc[0]))

,Split,Exists,Num Rows,Num Slides,Columns OK
0,train1,True,135946,7,True
1,train2,True,135946,3,True
2,train3,True,135946,3,True
3,train4,True,135946,1,True
4,train5,True,135946,1,True
5,train6,True,135946,1,True
6,train7,True,135946,1,True
7,train8,True,135946,5,True
8,train9,True,135946,3,True
9,train10,True,135946,3,True


Number of existing train part files: 10
Total rows across train1-train10: 1359460
Sum of slide counts across train1-train10: 28
Rows in final_df_train.csv: 146848
Slides in final_df_train.csv: 7


In [32]:
# =========================
# Cell 31: Output columns and XML geometry cache
# =========================

output_columns = ["filename", "slide_name", "x", "y"] + label_columns + ["filepath"]

xml_geometry_cache = {}

print("Output columns:")
print(output_columns)
print("Number of output columns:", len(output_columns))

Output columns:
['filename', 'slide_name', 'x', 'y', 'Irregular epithelial stratification', 'Loss of polarity of basal cells', 'Drop shaped rete ridges', 'Premature keratinization in single cells', 'Loss of epithelial cell cohesion', 'Abnormal variation in nuclear size', 'Abnormal variation in nuclear shape', 'Abnormal variation in cell size', 'Abnormal variation in cell shape', 'Increased N:C ratio', 'Increased number and size of nucleoli', 'Hyperchromasia', 'filepath']
Number of output columns: 17


In [33]:
# =========================
# Cell 32: Cached XML geometry loader
# =========================

def get_label_geometries_for_slide(slide_name, selected_labels):
    """
    Load XML polygons and build union annotation regions for one slide.
    Uses cache to avoid repeated XML parsing.
    """
    if slide_name in xml_geometry_cache:
        return xml_geometry_cache[slide_name]

    xml_path = XML_DIR / f"{slide_name}.xml"

    if not xml_path.exists():
        raise FileNotFoundError(f"XML file not found for slide {slide_name}: {xml_path}")

    label_to_polygons, annotation_summary_df, expanded_annotation_df = parse_xml_polygons(
        xml_path,
        selected_labels=selected_labels
    )

    label_to_geometry = build_label_union_geometries(
        label_to_polygons,
        selected_labels=selected_labels
    )

    xml_geometry_cache[slide_name] = {
        "label_to_polygons": label_to_polygons,
        "label_to_geometry": label_to_geometry,
        "annotation_summary_df": annotation_summary_df,
        "expanded_annotation_df": expanded_annotation_df,
    }

    return xml_geometry_cache[slide_name]

In [34]:
# =========================
# Cell 33: Generate one soft-label CSV
# =========================

def generate_soft_label_csv_for_file(
    input_csv_path,
    output_csv_path,
    split_name,
    selected_labels,
    tile_size=224
):
    """
    Generate one union-area soft-label CSV from one Kee hard-label CSV.

    Output columns:
    filename, slide_name, x, y, 12 soft-label columns, filepath
    """
    print("=" * 100)
    print("Generating soft-label CSV")
    print("Split:", split_name)
    print("Input:", input_csv_path)
    print("Output:", output_csv_path)

    df = pd.read_csv(input_csv_path)
    df = fix_filepath(df)

    missing_columns = [col for col in output_columns if col not in df.columns]
    if len(missing_columns) > 0:
        raise ValueError(f"Missing required columns in {input_csv_path}: {missing_columns}")

    df = df.copy()
    df["__row_order"] = np.arange(len(df))

    slide_soft_dfs = []
    slide_summary_rows = []

    slide_names = df["slide_name"].drop_duplicates().tolist()

    print("Number of rows:", len(df))
    print("Number of slides:", len(slide_names))
    print("Slides:", slide_names)

    for slide_name in slide_names:
        print("-" * 100)
        print("Processing slide:", slide_name)

        slide_df = df[df["slide_name"] == slide_name].copy().reset_index(drop=True)

        cached = get_label_geometries_for_slide(
            slide_name=slide_name,
            selected_labels=selected_labels
        )

        label_to_geometry = cached["label_to_geometry"]
        label_to_polygons = cached["label_to_polygons"]

        soft_df = generate_soft_labels_for_slide_union(
            slide_df=slide_df,
            label_to_geometry=label_to_geometry,
            selected_labels=selected_labels,
            tile_size=tile_size,
            verbose=False
        )

        check_df = compare_soft_to_hard(
            hard_df=slide_df,
            soft_df=soft_df,
            selected_labels=selected_labels,
            threshold=0.5
        )

        num_tiles = len(slide_df)
        total_comparisons = num_tiles * len(selected_labels)
        total_mismatch = int(check_df["Mismatch Count"].sum())

        slide_summary_rows.append({
            "Split": split_name,
            "Slide": slide_name,
            "Num Tiles": num_tiles,
            "Total Hard Positive Count": int(check_df["Hard Positive Count"].sum()),
            "Total Soft > 0 Count": int(check_df["Soft > 0 Count"].sum()),
            "Total Boundary Count (0<soft<0.5)": int(check_df["Boundary Count (0<soft<0.5)"].sum()),
            "Total Soft>=0.5 Positive Count": int(check_df["Soft>=0.5 Positive Count"].sum()),
            "Total Mismatch Count": total_mismatch,
            "Overall Match Rate": 1 - (total_mismatch / total_comparisons),
            "Min Feature Match Rate": float(check_df["Match Rate"].min()),
            "Max Soft Value": float(check_df["Soft Max Value"].max()),
            "XML Labels With Polygons": int(sum(len(label_to_polygons[label]) > 0 for label in selected_labels))
        })

        slide_soft_dfs.append(soft_df)

    output_df = pd.concat(slide_soft_dfs, ignore_index=True)

    output_df = output_df.sort_values("__row_order").reset_index(drop=True)

    output_df = output_df[output_columns]

    for label in selected_labels:
        output_df[label] = output_df[label].astype(np.float32)

    output_csv_path.parent.mkdir(parents=True, exist_ok=True)

    output_df.to_csv(output_csv_path, index=False)

    summary_df = pd.DataFrame(slide_summary_rows)

    print("Saved soft-label CSV:", output_csv_path)
    print("Output shape:", output_df.shape)

    return output_df, summary_df

In [37]:
# =========================
# Cell 34: Pilot generation for train1
# =========================

pilot_input_path = RP50_DIR / "final_df_train1.csv"
pilot_output_path = SOFT_LABEL_DIR / "final_df_train1_soft_overlap.csv"

train1_soft_df, train1_summary_df = generate_soft_label_csv_for_file(
    input_csv_path=pilot_input_path,
    output_csv_path=pilot_output_path,
    split_name="train1",
    selected_labels=label_columns,
    tile_size=TILE_SIZE
)

display(train1_summary_df)

print("train1_soft_df shape:", train1_soft_df.shape)
print("Saved:", pilot_output_path)

Generating soft-label CSV
Split: train1
Input: /home/user/jiangjie/Jiangjie_Project/data/ResearchProject_50/final_df_train1.csv
Output: /home/user/jiangjie/Jiangjie_Project/data/ResearchProject_soft_labels/final_df_train1_soft_overlap.csv
Number of rows: 135946
Number of slides: 7
Slides: ['266-DP-14', '459-DP-16', '504-DP-06', '52-DP-14', '568-DP-17-B', '597-DP-09', '680-DP-17-SRI-R']
----------------------------------------------------------------------------------------------------
Processing slide: 266-DP-14
----------------------------------------------------------------------------------------------------
Processing slide: 459-DP-16
----------------------------------------------------------------------------------------------------
Processing slide: 504-DP-06
----------------------------------------------------------------------------------------------------
Processing slide: 52-DP-14
------------------------------------------------------------------------------------------------

,Split,Slide,Num Tiles,Total Hard Positive Count,Total Soft > 0 Count,Total Boundary Count (0<soft<0.5),Total Soft>=0.5 Positive Count,Total Mismatch Count,Overall Match Rate,Min Feature Match Rate,Max Soft Value,XML Labels With Polygons
0,train1,266-DP-14,8855,914,1171,257,914,0,1.000000,1.000000,1.0,5
1,train1,459-DP-16,27459,38,71,33,38,0,1.000000,1.000000,1.0,3
2,train1,504-DP-06,26286,3217,3956,736,3220,3,0.999990,0.999924,1.0,10
3,train1,52-DP-14,22876,107,202,95,107,0,1.000000,1.000000,1.0,6
4,train1,568-DP-17-B,21452,296,619,322,297,1,0.999996,0.999953,1.0,6
5,train1,597-DP-09,24286,3961,4671,705,3966,5,0.999983,0.999835,1.0,6
6,train1,680-DP-17-SRI-R,4732,519,670,151,519,0,1.000000,1.000000,1.0,11


train1_soft_df shape: (135946, 17)
Saved: /home/user/jiangjie/Jiangjie_Project/data/ResearchProject_soft_labels/final_df_train1_soft_overlap.csv


In [38]:
# =========================
# Cell 35: Generate soft-label CSVs for train2-train10
# =========================

all_generation_summary_dfs = []

# Keep train1 summary from pilot generation
all_generation_summary_dfs.append(train1_summary_df)

for i in range(2, 11):
    split_name = f"train{i}"
    input_path = RP50_DIR / f"final_df_train{i}.csv"
    output_path = SOFT_LABEL_DIR / f"final_df_train{i}_soft_overlap.csv"

    soft_df_i, summary_df_i = generate_soft_label_csv_for_file(
        input_csv_path=input_path,
        output_csv_path=output_path,
        split_name=split_name,
        selected_labels=label_columns,
        tile_size=TILE_SIZE
    )

    all_generation_summary_dfs.append(summary_df_i)

    print(f"Finished {split_name}")
    print("Output shape:", soft_df_i.shape)
    print("Saved:", output_path)

    # Free memory
    del soft_df_i

Generating soft-label CSV
Split: train2
Input: /home/user/jiangjie/Jiangjie_Project/data/ResearchProject_50/final_df_train2.csv
Output: /home/user/jiangjie/Jiangjie_Project/data/ResearchProject_soft_labels/final_df_train2_soft_overlap.csv
Number of rows: 135946
Number of slides: 3
Slides: ['680-DP-17-SRI-R', '707-DP-17-2', '876-DP-11']
----------------------------------------------------------------------------------------------------
Processing slide: 680-DP-17-SRI-R
----------------------------------------------------------------------------------------------------
Processing slide: 707-DP-17-2
----------------------------------------------------------------------------------------------------
Processing slide: 876-DP-11
Saved soft-label CSV: /home/user/jiangjie/Jiangjie_Project/data/ResearchProject_soft_labels/final_df_train2_soft_overlap.csv
Output shape: (135946, 17)
Finished train2
Output shape: (135946, 17)
Saved: /home/user/jiangjie/Jiangjie_Project/data/ResearchProject_soft_la

In [39]:
# =========================
# Cell 36: Generate soft-label CSVs for validation and test
# =========================

val_input_path = RP50_DIR / "final_df_val.csv"
val_output_path = SOFT_LABEL_DIR / "final_df_val_soft_overlap.csv"

val_soft_df, val_summary_df = generate_soft_label_csv_for_file(
    input_csv_path=val_input_path,
    output_csv_path=val_output_path,
    split_name="validation",
    selected_labels=label_columns,
    tile_size=TILE_SIZE
)

all_generation_summary_dfs.append(val_summary_df)

print("Finished validation")
print("Output shape:", val_soft_df.shape)
print("Saved:", val_output_path)

del val_soft_df


test_input_path = RP50_DIR / "final_df_test.csv"
test_output_path = SOFT_LABEL_DIR / "final_df_test_soft_overlap.csv"

test_soft_df, test_summary_df = generate_soft_label_csv_for_file(
    input_csv_path=test_input_path,
    output_csv_path=test_output_path,
    split_name="test",
    selected_labels=label_columns,
    tile_size=TILE_SIZE
)

all_generation_summary_dfs.append(test_summary_df)

print("Finished test")
print("Output shape:", test_soft_df.shape)
print("Saved:", test_output_path)

del test_soft_df

Generating soft-label CSV
Split: validation
Input: /home/user/jiangjie/Jiangjie_Project/data/ResearchProject_50/final_df_val.csv
Output: /home/user/jiangjie/Jiangjie_Project/data/ResearchProject_soft_labels/final_df_val_soft_overlap.csv
Number of rows: 198083
Number of slides: 6
Slides: ['D706-12-1-i', '168-DP-09', '895-DP-09', '1253-DP-12', '1318-DP-08', '1467-DP-12']
----------------------------------------------------------------------------------------------------
Processing slide: D706-12-1-i
----------------------------------------------------------------------------------------------------
Processing slide: 168-DP-09
----------------------------------------------------------------------------------------------------
Processing slide: 895-DP-09
----------------------------------------------------------------------------------------------------
Processing slide: 1253-DP-12
----------------------------------------------------------------------------------------------------
Processi

In [40]:
# =========================
# Cell 37: Merge train1-train10 soft-label CSVs into full train soft CSV
# =========================

train_soft_paths = [
    SOFT_LABEL_DIR / f"final_df_train{i}_soft_overlap.csv"
    for i in range(1, 11)
]

for path in train_soft_paths:
    print(path.name, "exists:", path.exists())

train_soft_dfs = []

for path in train_soft_paths:
    temp_df = pd.read_csv(path)
    train_soft_dfs.append(temp_df)

final_train_soft_df = pd.concat(train_soft_dfs, ignore_index=True)

final_train_soft_path = SOFT_LABEL_DIR / "final_df_train_soft_overlap.csv"
final_train_soft_df.to_csv(final_train_soft_path, index=False)

print("Saved full train soft CSV:", final_train_soft_path)
print("final_train_soft_df shape:", final_train_soft_df.shape)

del train_soft_dfs
del final_train_soft_df

final_df_train1_soft_overlap.csv exists: True
final_df_train2_soft_overlap.csv exists: True
final_df_train3_soft_overlap.csv exists: True
final_df_train4_soft_overlap.csv exists: True
final_df_train5_soft_overlap.csv exists: True
final_df_train6_soft_overlap.csv exists: True
final_df_train7_soft_overlap.csv exists: True
final_df_train8_soft_overlap.csv exists: True
final_df_train9_soft_overlap.csv exists: True
final_df_train10_soft_overlap.csv exists: True
Saved full train soft CSV: /home/user/jiangjie/Jiangjie_Project/data/ResearchProject_soft_labels/final_df_train_soft_overlap.csv
final_train_soft_df shape: (1359460, 17)


In [41]:
# =========================
# Cell 38: Save full generation summary
# =========================

soft_label_generation_summary_df = pd.concat(
    all_generation_summary_dfs,
    ignore_index=True
)

summary_output_path = SOFT_LABEL_DIR / "soft_label_generation_summary.csv"

soft_label_generation_summary_df.to_csv(summary_output_path, index=False)

print("Saved summary:", summary_output_path)
print("Summary shape:", soft_label_generation_summary_df.shape)

display(soft_label_generation_summary_df.head(20))

Saved summary: /home/user/jiangjie/Jiangjie_Project/data/ResearchProject_soft_labels/soft_label_generation_summary.csv
Summary shape: (40, 12)


,Split,Slide,Num Tiles,Total Hard Positive Count,Total Soft > 0 Count,Total Boundary Count (0<soft<0.5),Total Soft>=0.5 Positive Count,Total Mismatch Count,Overall Match Rate,Min Feature Match Rate,Max Soft Value,XML Labels With Polygons
0,train1,266-DP-14,8855,914,1171,257,914,0,1.000000,1.000000,1.0,5
1,train1,459-DP-16,27459,38,71,33,38,0,1.000000,1.000000,1.0,3
2,train1,504-DP-06,26286,3217,3956,736,3220,3,0.999990,0.999924,1.0,10
3,train1,52-DP-14,22876,107,202,95,107,0,1.000000,1.000000,1.0,6
4,train1,568-DP-17-B,21452,296,619,322,297,1,0.999996,0.999953,1.0,6
5,train1,597-DP-09,24286,3961,4671,705,3966,5,0.999983,0.999835,1.0,6
6,train1,680-DP-17-SRI-R,4732,519,670,151,519,0,1.000000,1.000000,1.0,11
7,train2,680-DP-17-SRI-R,4781,917,1115,197,918,1,0.999983,0.999791,1.0,11
8,train2,707-DP-17-2,40834,5414,6210,793,5417,3,0.999994,0.999951,1.0,9
9,train2,876-DP-11,90331,1698,2612,913,1699,1,0.999999,0.999989,1.0,10


In [43]:
# =========================
# Cell 39: Check final output files
# =========================

final_output_files = []

for i in range(1, 11):
    final_output_files.append(SOFT_LABEL_DIR / f"final_df_train{i}_soft_overlap.csv")

final_output_files.extend([
    SOFT_LABEL_DIR / "final_df_train_soft_overlap.csv",
    SOFT_LABEL_DIR / "final_df_val_soft_overlap.csv",
    SOFT_LABEL_DIR / "final_df_test_soft_overlap.csv",
    SOFT_LABEL_DIR / "soft_label_generation_summary.csv",
])

output_check_rows = []

for path in final_output_files:
    exists = path.exists()

    row = {
        "File": path.name,
        "Exists": exists,
        "Num Rows": None,
        "Num Columns": None,
        "Path": str(path),
    }

    if exists and path.suffix == ".csv":
        temp_header = pd.read_csv(path, nrows=0)
        row["Num Columns"] = len(temp_header.columns)

        if "summary" not in path.name:
            temp_count = pd.read_csv(path, usecols=["slide_name"])
            row["Num Rows"] = len(temp_count)
        else:
            temp_count = pd.read_csv(path)
            row["Num Rows"] = len(temp_count)

    output_check_rows.append(row)

final_output_check_df = pd.DataFrame(output_check_rows)

display(final_output_check_df)

,File,Exists,Num Rows,Num Columns,Path
0,final_df_train1_soft_overlap.csv,True,135946,17,/home/user/jiangjie/Jiangjie_Project/data/Rese...
1,final_df_train2_soft_overlap.csv,True,135946,17,/home/user/jiangjie/Jiangjie_Project/data/Rese...
2,final_df_train3_soft_overlap.csv,True,135946,17,/home/user/jiangjie/Jiangjie_Project/data/Rese...
3,final_df_train4_soft_overlap.csv,True,135946,17,/home/user/jiangjie/Jiangjie_Project/data/Rese...
4,final_df_train5_soft_overlap.csv,True,135946,17,/home/user/jiangjie/Jiangjie_Project/data/Rese...
5,final_df_train6_soft_overlap.csv,True,135946,17,/home/user/jiangjie/Jiangjie_Project/data/Rese...
6,final_df_train7_soft_overlap.csv,True,135946,17,/home/user/jiangjie/Jiangjie_Project/data/Rese...
7,final_df_train8_soft_overlap.csv,True,135946,17,/home/user/jiangjie/Jiangjie_Project/data/Rese...
8,final_df_train9_soft_overlap.csv,True,135946,17,/home/user/jiangjie/Jiangjie_Project/data/Rese...
9,final_df_train10_soft_overlap.csv,True,135946,17,/home/user/jiangjie/Jiangjie_Project/data/Rese...


In [44]:
# =========================
# Cell 40: Final QA function for generated soft-label CSVs
# =========================

def final_qa_for_soft_csv(csv_path, expected_rows=None, expected_columns=17, selected_labels=None):
    """
    Final QA for one generated soft-label CSV.

    Checks:
    1. File exists
    2. Row count
    3. Column count
    4. Required columns
    5. NaN in soft-label columns
    6. Min/max of soft-label columns
    7. Out-of-range values outside [0, 1]
    """
    csv_path = Path(csv_path)

    row = {
        "File": csv_path.name,
        "Exists": csv_path.exists(),
        "Expected Rows": expected_rows,
        "Actual Rows": None,
        "Rows OK": None,
        "Expected Columns": expected_columns,
        "Actual Columns": None,
        "Columns OK": None,
        "Missing Columns": None,
        "Any NaN in Labels": None,
        "Any Label < 0": None,
        "Any Label > 1": None,
        "Global Label Min": None,
        "Global Label Max": None,
    }

    if not csv_path.exists():
        return row

    header_df = pd.read_csv(csv_path, nrows=0)
    columns = header_df.columns.tolist()

    row["Actual Columns"] = len(columns)
    row["Columns OK"] = len(columns) == expected_columns

    required_columns = ["filename", "slide_name", "x", "y"] + selected_labels + ["filepath"]
    missing_columns = [col for col in required_columns if col not in columns]
    row["Missing Columns"] = missing_columns

    # Read only needed columns
    df = pd.read_csv(csv_path, usecols=required_columns)

    row["Actual Rows"] = len(df)

    if expected_rows is not None:
        row["Rows OK"] = len(df) == expected_rows
    else:
        row["Rows OK"] = True

    label_df = df[selected_labels]

    row["Any NaN in Labels"] = bool(label_df.isna().any().any())
    row["Any Label < 0"] = bool((label_df < 0).any().any())
    row["Any Label > 1"] = bool((label_df > 1).any().any())

    row["Global Label Min"] = float(label_df.min().min())
    row["Global Label Max"] = float(label_df.max().max())

    return row

In [45]:
# =========================
# Cell 41: Run final QA for all generated soft-label CSVs
# =========================

expected_row_counts = {}

for i in range(1, 11):
    expected_row_counts[f"final_df_train{i}_soft_overlap.csv"] = 135946

expected_row_counts["final_df_train_soft_overlap.csv"] = 1359460
expected_row_counts["final_df_val_soft_overlap.csv"] = 198083
expected_row_counts["final_df_test_soft_overlap.csv"] = 314284

qa_files = []

for i in range(1, 11):
    qa_files.append(SOFT_LABEL_DIR / f"final_df_train{i}_soft_overlap.csv")

qa_files.extend([
    SOFT_LABEL_DIR / "final_df_train_soft_overlap.csv",
    SOFT_LABEL_DIR / "final_df_val_soft_overlap.csv",
    SOFT_LABEL_DIR / "final_df_test_soft_overlap.csv",
])

qa_rows = []

for csv_path in qa_files:
    print("Checking:", csv_path.name)

    qa_row = final_qa_for_soft_csv(
        csv_path=csv_path,
        expected_rows=expected_row_counts[csv_path.name],
        expected_columns=17,
        selected_labels=label_columns
    )

    qa_rows.append(qa_row)

final_qa_df = pd.DataFrame(qa_rows)

display(final_qa_df)

Checking: final_df_train1_soft_overlap.csv
Checking: final_df_train2_soft_overlap.csv
Checking: final_df_train3_soft_overlap.csv
Checking: final_df_train4_soft_overlap.csv
Checking: final_df_train5_soft_overlap.csv
Checking: final_df_train6_soft_overlap.csv
Checking: final_df_train7_soft_overlap.csv
Checking: final_df_train8_soft_overlap.csv
Checking: final_df_train9_soft_overlap.csv
Checking: final_df_train10_soft_overlap.csv
Checking: final_df_train_soft_overlap.csv
Checking: final_df_val_soft_overlap.csv
Checking: final_df_test_soft_overlap.csv


,File,Exists,Expected Rows,Actual Rows,Rows OK,Expected Columns,Actual Columns,Columns OK,Missing Columns,Any NaN in Labels,Any Label < 0,Any Label > 1,Global Label Min,Global Label Max
0,final_df_train1_soft_overlap.csv,True,135946,135946,True,17,17,True,[],False,False,False,0.0,1.0
1,final_df_train2_soft_overlap.csv,True,135946,135946,True,17,17,True,[],False,False,False,0.0,1.0
2,final_df_train3_soft_overlap.csv,True,135946,135946,True,17,17,True,[],False,False,False,0.0,1.0
3,final_df_train4_soft_overlap.csv,True,135946,135946,True,17,17,True,[],False,False,False,0.0,0.0
4,final_df_train5_soft_overlap.csv,True,135946,135946,True,17,17,True,[],False,False,False,0.0,0.0
5,final_df_train6_soft_overlap.csv,True,135946,135946,True,17,17,True,[],False,False,False,0.0,0.0
6,final_df_train7_soft_overlap.csv,True,135946,135946,True,17,17,True,[],False,False,False,0.0,1.0
7,final_df_train8_soft_overlap.csv,True,135946,135946,True,17,17,True,[],False,False,False,0.0,1.0
8,final_df_train9_soft_overlap.csv,True,135946,135946,True,17,17,True,[],False,False,False,0.0,1.0
9,final_df_train10_soft_overlap.csv,True,135946,135946,True,17,17,True,[],False,False,False,0.0,1.0


In [46]:
# =========================
# Cell 42: Final QA pass/fail summary
# =========================

qa_pass_conditions = {
    "All files exist": final_qa_df["Exists"].all(),
    "All row counts OK": final_qa_df["Rows OK"].all(),
    "All column counts OK": final_qa_df["Columns OK"].all(),
    "No missing columns": final_qa_df["Missing Columns"].apply(lambda x: len(x) == 0).all(),
    "No NaN in labels": (~final_qa_df["Any NaN in Labels"]).all(),
    "No label < 0": (~final_qa_df["Any Label < 0"]).all(),
    "No label > 1": (~final_qa_df["Any Label > 1"]).all(),
}

qa_pass_summary_df = pd.DataFrame([
    {"Check": key, "Passed": value}
    for key, value in qa_pass_conditions.items()
])

display(qa_pass_summary_df)

all_passed = all(qa_pass_conditions.values())

print("All final QA checks passed:", all_passed)

if all_passed:
    print("Soft-label CSV generation is COMPLETE and PASSED final QA.")
else:
    print("Some QA checks failed. Please inspect final_qa_df.")

,Check,Passed
0,All files exist,True
1,All row counts OK,True
2,All column counts OK,True
3,No missing columns,True
4,No NaN in labels,True
5,No label < 0,True
6,No label > 1,True


All final QA checks passed: True
Soft-label CSV generation is COMPLETE and PASSED final QA.


In [47]:
# =========================
# Cell 43: Save final QA results
# =========================

final_qa_path = SOFT_LABEL_DIR / "soft_label_final_qa.csv"
qa_pass_summary_path = SOFT_LABEL_DIR / "soft_label_final_qa_pass_summary.csv"

final_qa_df.to_csv(final_qa_path, index=False)
qa_pass_summary_df.to_csv(qa_pass_summary_path, index=False)

print("Saved:")
print(final_qa_path)
print(qa_pass_summary_path)

Saved:
/home/user/jiangjie/Jiangjie_Project/data/ResearchProject_soft_labels/soft_label_final_qa.csv
/home/user/jiangjie/Jiangjie_Project/data/ResearchProject_soft_labels/soft_label_final_qa_pass_summary.csv
